# debug

In [1]:
import asyncio
import threading
from playwright.async_api import async_playwright

captured_responses = []
log = []

async def handle_response(response):
    url = response.url
    # chart data biasanya lewat request XHR/fetch yang balikin JSON
    if response.request.resource_type in ("xhr", "fetch"):
        try:
            ct = response.headers.get("content-type", "")
            if "json" in ct:
                text = await response.text()
                captured_responses.append((url, text))
        except Exception:
            pass

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        page.on("response", handle_response)

        await page.goto(
            "https://tradingeconomics.com/indonesia/manufacturing-pmi",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(8000)

        # coba klik tombol "1Y" biar chart load data 1 tahun (defaultnya mungkin cuma 6M)
        try:
            btn_1y = page.locator('text="1Y"').first
            if await btn_1y.count() > 0:
                await btn_1y.click()
                await page.wait_for_timeout(4000)
                log.append("klik 1Y OK")
        except Exception as e:
            log.append(f"klik 1Y gagal: {e}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)

print(f"\n=== {len(captured_responses)} response JSON ketangkep ===")
for url, text in captured_responses:
    print(url, "->", len(text), "karakter")

=== LOG ===
klik 1Y OK

=== 3 response JSON ketangkep ===
https://d3ii0wo49og5mi.cloudfront.net/economics/indonesiamanpmi?n=12&v=20260701004800 -> 538 karakter
https://chat.tradingeconomics.com/api/feature-flags -> 45 karakter
https://d3ii0wo49og5mi.cloudfront.net/economics/indonesiamanpmi?&span=1y&v=20260701004800 -> 550 karakter


In [2]:
for url, text in captured_responses:
    if "indonesiamanpmi" in url:
        print(url)
        print(text)
        print("---")

https://d3ii0wo49og5mi.cloudfront.net/economics/indonesiamanpmi?n=12&v=20260701004800
"a/lpZGluZ2VjbBO9oOO4QzlmiDZ7E6K4AUur9wDMeyo7aGYkbl01I6DWv/ck+c9OibToYBZcBog8g4FczcEiHOsBMwoqgxdHNu1QJMofHIqPv/ah1dcIB5lDGMfBpKtQvNNpUjMtba7904aflw9m5oi94J0cXdRJL8qU/qZgYIPb64cYH58RzYqLt96ipj6SC+h3YO4O2COIhX1bV4aBFFZhKYw1Y7D5uEKF+ZdHqsqS3ZpMCB3D5UelK0D/PSQ3QzGp+MSLrtdXlOemTqLF8YxD1Qmk8zw/3b43yDF2Ktf6JjcLqOPKNPdRPJAu2muOaCThketFIyvdfgYG0bstAJD0luCSK6BDijWrFAvUnP/kUqR6A9Ym73PWCQ/aX//Jb7kYrrr5xhcovPcKh6dn1kFeXBrWperIIcoZQ7tV6cE2QE3mYg8Et3/GTefgYNhZLu1NWK9dkD7E+v5as1gs/vvbzWAiXJBW4/gBjOTt4XzAaO9WawHTjsa8U8IZUofV4fKLmHk+Xrwy1nvpND9iZGk="
---
https://d3ii0wo49og5mi.cloudfront.net/economics/indonesiamanpmi?&span=1y&v=20260701004800
"a/lpZGluZ2VjbBO9JuK4Qz1jiN5gkqO+AUsranaGMqBMCsQ+l/GLTDB7PBcGdnQtkxLybManRdw7t7ShV1yneCty3HBtPq5ooG9nNAmzvl5QN7MGuh41Ni5r9I9+XEtQXtMbUnMVY65NtpCJigRjYKrpaJScCxpZ7+u9KDpkKHEdCaF+ka79xGTWnzbzZLojizHW5Il8q8qy14hpZ1OMkWoYa2X/ay4hShkourTDSvOG9KeGXgy3NerETUPRfQ3j5Q3mv2hZ0tAfZSNhmn75Rgsx

In [6]:
import asyncio
import threading
from playwright.async_api import async_playwright

log = []
elements_found = []

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://tradingeconomics.com/indonesia/manufacturing-pmi",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(8000)

        # cari posisi Y dari tombol "1Y" sebagai patokan
        anchor_y = await page.evaluate("""
            () => {
                const all = document.querySelectorAll('*');
                for (const el of all) {
                    if (el.textContent.trim() === '1Y' && el.children.length === 0) {
                        const rect = el.getBoundingClientRect();
                        return rect.y;
                    }
                }
                return null;
            }
        """)
        log.append(f"Posisi Y tombol 1Y: {anchor_y}")

        # cari semua elemen clickable di sekitar y itu (+- 100px), di seluruh halaman
        elements = await page.evaluate(f"""
            () => {{
                const anchorY = {anchor_y if anchor_y is not None else 0};
                const els = document.querySelectorAll('button, [role="button"], svg, [class*="menu"], [class*="dots"], [class*="export"], [class*="icon"], a');
                const results = [];
                for (const el of els) {{
                    const rect = el.getBoundingClientRect();
                    if (Math.abs(rect.y - anchorY) < 100 && rect.width > 0) {{
                        results.push({{
                            tag: el.tagName,
                            class: (el.className && typeof el.className === 'string') ? el.className : String(el.className),
                            id: el.id,
                            x: Math.round(rect.x),
                            y: Math.round(rect.y),
                            w: Math.round(rect.width),
                            h: Math.round(rect.height),
                            text: el.textContent.trim().slice(0, 30),
                        }});
                    }}
                }}
                return results;
            }}
        """)
        elements_found.extend(elements)

        await page.screenshot(path="chart_area2.png")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)

print(f"\n=== {len(elements_found)} elemen di sekitar toolbar chart ===")
for e in elements_found:
    print(e)

=== LOG ===
Posisi Y tombol 1Y: 413.03125

=== 25 elemen di sekitar toolbar chart ===
{'tag': 'A', 'class': 'navbar-te-link mx-1', 'id': '', 'x': 1317, 'y': 385, 'w': 63, 'h': 17, 'text': 'Features'}
{'tag': 'A', 'class': 'navbar-te-link mx-1', 'id': '', 'x': 1401, 'y': 385, 'w': 36, 'h': 17, 'text': 'Docs'}
{'tag': 'A', 'class': 'navbar-te-link mx-1', 'id': '', 'x': 1459, 'y': 385, 'w': 73, 'h': 17, 'text': 'Developer'}
{'tag': 'A', 'class': 'user-item list-group-item', 'id': '', 'x': 1297, 'y': 439, 'w': 292, 'h': 54, 'text': 'Already a user? Login'}
{'tag': 'DIV', 'class': 'list-group list-group-user dk-switch-right-menu', 'id': '', 'x': 1297, 'y': 504, 'w': 292, 'h': 57, 'text': ''}
{'tag': 'A', 'class': 'navmobile-link nav-link', 'id': '', 'x': -321, 'y': 351, 'w': 316, 'h': 42, 'text': 'Inflation Rate'}
{'tag': 'A', 'class': 'navmobile-link nav-link', 'id': '', 'x': -321, 'y': 394, 'w': 316, 'h': 42, 'text': 'Unemployment Rate'}
{'tag': 'A', 'class': 'navmobile-link nav-link', 'i

# final code

In [4]:
import asyncio
import threading
import re
from playwright.async_api import async_playwright

log = []
download_path = {}

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://tradingeconomics.com/indonesia/manufacturing-pmi",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(8000)
        log.append("halaman ke-load")

        # klik "1Y" dan verifikasi beneran aktif
        try:
            btn_1y = page.locator('text="1Y"').first
            await btn_1y.click(timeout=5000)
            await page.wait_for_timeout(2000)

            is_active = await btn_1y.evaluate("""
                (el) => {
                    const style = window.getComputedStyle(el);
                    return style.backgroundColor !== 'rgba(0, 0, 0, 0)' && style.backgroundColor !== 'transparent';
                }
            """)
            log.append(f"Tombol 1Y aktif? {is_active}")
            await page.wait_for_timeout(2000)
        except Exception as e:
            log.append(f"klik 1Y gagal: {e}")

        # klik ikon titik tiga (menu export chart)
        try:
            menu_candidates = [
                'button:has(svg):near(:text("Compare"))',
                '[class*="dots"]',
                '[class*="menu-icon"]',
                'button >> nth=-1',
            ]
            clicked = False
            for sel in menu_candidates:
                try:
                    el = page.locator(sel).last
                    if await el.count() > 0:
                        await el.click(timeout=3000)
                        clicked = True
                        log.append(f"klik menu pakai selector: {sel}")
                        break
                except Exception:
                    continue
            if not clicked:
                log.append("Gagal klik menu titik-tiga lewat semua selector")
        except Exception as e:
            log.append(f"error klik menu: {e}")

        await page.wait_for_timeout(1000)

        # klik "SVG Image" dan tangkep download-nya
        try:
            async with page.expect_download(timeout=15000) as download_info:
                svg_option = page.locator('text="SVG Image"').first
                await svg_option.click()

            download = await download_info.value
            save_path = "pmi_chart.svg"
            await download.save_as(save_path)
            download_path["path"] = save_path
            log.append(f"SVG tersimpan: {save_path}")
        except Exception as e:
            log.append(f"gagal download SVG: {e}")
            await page.screenshot(path="debug_pmi.png")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR FATAL: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)
print(download_path)

=== LOG ===
halaman ke-load
Tombol 1Y aktif? True
Gagal klik menu titik-tiga lewat semua selector
gagal download SVG: Locator.click: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("text=\"SVG Image\"").first

{}


In [7]:
import asyncio
import threading
from playwright.async_api import async_playwright

log = []
download_path = {}

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
            accept_downloads=True,
        )
        page = await context.new_page()

        await page.goto(
            "https://tradingeconomics.com/indonesia/manufacturing-pmi",
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(8000)
        log.append("halaman ke-load")

        # klik "1Y"
        btn_1y = page.locator('a.hawk-chartOptions-datePicker-cnt-btn:has-text("1Y")')
        await btn_1y.click()
        await page.wait_for_timeout(3000)
        log.append("klik 1Y OK")

        # klik menu titik-tiga (auxExportingBtn)
        menu_btn = page.locator('.auxExportingBtn')
        await menu_btn.click()
        log.append("klik menu titik-tiga OK")
        await page.wait_for_timeout(1000)

        # klik SVG Image, tangkep download
        async with page.expect_download(timeout=15000) as download_info:
            svg_option = page.locator('text="SVG Image"').first
            await svg_option.click()

        download = await download_info.value
        save_path = "pmi_chart.svg"
        await download.save_as(save_path)
        download_path["path"] = save_path
        log.append(f"SVG tersimpan: {save_path}")

        await browser.close()

def run_in_thread(coro_func):
    def runner():
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(coro_func())
        except Exception as e:
            log.append(f"ERROR: {e}")
        finally:
            loop.close()
    t = threading.Thread(target=runner)
    t.start()
    t.join()

run_in_thread(run)

print("=== LOG ===")
for l in log:
    print(l)
print(download_path)

=== LOG ===
halaman ke-load
klik 1Y OK
klik menu titik-tiga OK
SVG tersimpan: pmi_chart.svg
{'path': 'pmi_chart.svg'}


In [ ]:
def parse_pmi_svg(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    label_pattern = re.compile(r'<text x="573"[^>]*\sy="([\d.]+)"[^>]*><tspan[^>]*>(\d+)</tspan></text>')
    label_matches = label_pattern.findall(content)
    y_pixels = [float(y) for y, v in label_matches]
    y_values = [float(v) for y, v in label_matches]

    n = len(y_pixels)
    mean_x = sum(y_pixels)/n
    mean_y = sum(y_values)/n
    m = sum((x-mean_x)*(y-mean_y) for x,y in zip(y_pixels,y_values)) / sum((x-mean_x)**2 for x in y_pixels)
    b = mean_y - m*mean_x

    transform_match = re.search(r'class="highcharts-series highcharts-series-0[^"]*"[^>]*transform="translate\(([\d.-]+),\s*([\d.-]+)\)', content)
    offset_x = float(transform_match.group(1)) if transform_match else 0
    offset_y = float(transform_match.group(2)) if transform_match else 0

    bar_pattern = re.compile(r'<rect x="([\d.-]+)" y="([\d.-]+)" width="([\d.-]+)" height="([\d.-]+)" fill="[^"]*" opacity="1" class="highcharts-point">')
    bars = bar_pattern.findall(content)
    bars = [(float(x), float(y), float(w), float(h)) for x,y,w,h in bars]
    bars.sort(key=lambda b: b[0])

    values = []
    for x, y, w, h in bars:
        y_abs = y + offset_y
        val = m * y_abs + b
        x_center = x + w/2 + offset_x
        values.append((x_center, round(val, 2)))

    xlabel_pattern = re.compile(r'<g class="highcharts-axis-labels highcharts-xaxis-labels"[^>]*>(.*?)</g>', re.DOTALL)
    xlabel_block = xlabel_pattern.search(content).group(1)
    xlabel_items = re.findall(r'<text x="([\d.]+)"[^>]*>([^<]+)</text>', xlabel_block)
    xlabel_items = [(float(x), txt) for x, txt in xlabel_items]

    bulan_list = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

    bar_x_centers = [v[0] for v in values]
    anchor_idx_to_label = {}
    for x, txt in xlabel_items:
        idx = min(range(len(bar_x_centers)), key=lambda i: abs(bar_x_centers[i]-x))
        anchor_idx_to_label[idx] = txt

    year_anchor_idx = None
    year_anchor_val = None
    for idx, txt in anchor_idx_to_label.items():
        if re.fullmatch(r'\d{4}', txt):
            year_anchor_idx = idx
            year_anchor_val = int(txt)
            break

    if year_anchor_idx is None:
        raise ValueError("Gak ketemu anchor tahun, gak bisa rekonstruksi bulan otomatis")

    results = []
    for i, (x_center, val) in enumerate(values):
        month_offset = i - year_anchor_idx
        month_num = (0 + month_offset) % 12
        year = year_anchor_val + (0 + month_offset) // 12
        bulan_nama = bulan_list[month_num]
        results.append((f"{bulan_nama} {year}", val))

    return results


hasil = parse_pmi_svg(download_path["path"])
for bulan, val in hasil:
    print(f"{bulan}: {val}")